# **Libraries**

In [17]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

import folium

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

from math import radians, sin, cos, sqrt, atan2

# **Data Collection**

In [18]:
DATABASE_URL = "sqlite:///C:/Users/Xavier/Documents/Git Clone/samling-ai/backend/samling.db"
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)
db = Session()

In [19]:
latest_batch = pd.read_sql("""
SELECT MAX(forecast_batch_id) AS batch
FROM volume_predictions
""", engine).iloc[0]["batch"]

latest_batch

'batch_20260712_161644'

In [20]:
forecast_df = pd.read_sql(f"""
SELECT *
FROM volume_predictions
WHERE forecast_batch_id = '{latest_batch}'
""", engine)

forecast_df.head()

,id,created_at,predicted_volume_percentage,forecast_batch_id,kecamatan,tps_id,priority_rank,prediction_status,model_version
0,1,2026-07-12 09:16:44,76.573619,batch_20260712_161644,Mampang Prapatan,88,None,WARNING,forecast_waste_volume_model


In [21]:
zones = pd.read_sql("""
SELECT
    id,
    kecamatan,
    latitude,
    longitude
FROM zones
""", engine)

forecast_df = forecast_df.merge(
    zones,
    left_on="tps_id",
    right_on="id"
)

forecast_df.head()

,id_x,created_at,predicted_volume_percentage,forecast_batch_id,kecamatan_x,tps_id,priority_rank,prediction_status,model_version,id_y,kecamatan_y,latitude,longitude
0,1,2026-07-12 09:16:44,76.573619,batch_20260712_161644,Mampang Prapatan,88,None,WARNING,forecast_waste_volume_model,88,Mampang Prapatan,-6.241685,106.80816


In [22]:
forecast_df = forecast_df[
    forecast_df["prediction_status"] != "NORMAL"
]

forecast_df = forecast_df.sort_values(
    "predicted_volume_percentage",
    ascending=False
)

forecast_df

,id_x,created_at,predicted_volume_percentage,forecast_batch_id,kecamatan_x,tps_id,priority_rank,prediction_status,model_version,id_y,kecamatan_y,latitude,longitude
0,1,2026-07-12 09:16:44,76.573619,batch_20260712_161644,Mampang Prapatan,88,None,WARNING,forecast_waste_volume_model,88,Mampang Prapatan,-6.241685,106.80816
